# Module 28 — Claims Analytics Foundations

This notebook introduces CMS Medicare synthetic claims data using CMS DE-SynPUF.

Goal:
Build beginner-friendly claims analytics using realistic public Medicare claims files.

Main dataset:
CMS 2008–2010 Data Entrepreneurs’ Synthetic Public Use File (DE-SynPUF)

Focus:
- Beneficiary demographics
- Inpatient claims
- Outpatient claims
- Carrier claims
- Prescription drug events
- Cost and utilization analytics

# Module 28 — Claims Analytics Foundations

Dataset:
CMS DE-SynPUF Sample 1

Current task:
Verify raw uploaded files before ingestion.

Why:

Checking raw files is a standard data engineering practice before building Bronze tables.

## Step 1 — Verify uploaded CMS raw file

Before loading data, confirm the raw file exists.

This is standard data engineering practice before Bronze ingestion.

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/healthcare_catalog/bronze/raw_claims_files/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/176541_DE1_0_2008_Beneficiary_Summary_File_Sample_1.zip,176541_DE1_0_2008_Beneficiary_Summary_File_Sample_1.zip,3119160,1779405806000


## Step 2 — Inspect compressed CMS file

CMS files are often downloaded as ZIP archives.

Before reading data, we check what files exist inside the archive.

In [0]:
import zipfile

zip_path = "/dbfs/Volumes/healthcare_catalog/bronze/raw_claims_files/176541_DE1_0_2008_Beneficiary_Summary_File_Sample_1.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    print(z.namelist())

---------------------------------------------------------------------------
OSError                                   Traceback (most recent call last)
File <command-7177429497791344>, line 5
      1 import zipfile
      3 zip_path = "/dbfs/Volumes/healthcare_catalog/bronze/raw_claims_files/176541_DE1_0_2008_Beneficiary_Summary_File_Sample_1.zip"
----> 5 with zipfile.ZipFile(zip_path, 'r') as z:
      6     print(z.namelist())

File /usr/lib/python3.12/zipfile/__init__.py:1347, in ZipFile.__init__(self, file, mode, compression, allowZip64, compresslevel, strict_timestamps, metadata_encoding)
   1345 while True:
   1346     try:
-> 1347         self.fp = io.open(file, filemode)
   1348     except OSError:
   1349         if filemode in modeDict:

OSError: [Errno 5] Input/output error: '/dbfs/Volumes/healthcare_catalog/bronze/raw_claims_files/176541_DE1_0_2008_Beneficiary_Summary_File_Sample_1.zip'

In [0]:
dbutils.fs.cp(
    "dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/176541_DE1_0_2008_Beneficiary_Summary_File_Sample_1.zip",
    "file:/tmp/cms_beneficiary.zip"
)

---------------------------------------------------------------------------
ExecutionError                            Traceback (most recent call last)
File <command-7177429497791345>, line 1
----> 1 dbutils.fs.cp(
      2     "dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/176541_DE1_0_2008_Beneficiary_Summary_File_Sample_1.zip",
      3     "file:/tmp/cms_beneficiary.zip"
      4 )

File /databricks/python_shell/lib/dbruntime/remotefshandler/RemoteFsHandler.py:56, in prettify_exception_message.<locals>.f_with_exception_handling(*args, **kwargs)
     52     pass
     54 error_exception = ExecutionError(str(e))
---> 56 raise patch_exception_with_error_details(
     57     error_exception,
     58     DriverErrorCode.REMOTE_FS_HANDLER_EXECUTION_ERROR  # type: ignore[attr-defined]
     59 ) from None

ExecutionError: (com.databricks.backend.daemon.driver.LocalFilesystemAccessDeniedException) Cannot access non /Workspace local filesystem path: file:/tmp/cms_beneficiary.zip

JVM 

In [0]:
import os

volume_path = "/Volumes/healthcare_catalog/bronze/raw_claims_files"

print(os.listdir(volume_path))

['176541_DE1_0_2008_Beneficiary_Summary_File_Sample_1.zip']


## Step 5 — Create extraction folder for CMS files

Current status:

We uploaded the CMS beneficiary ZIP file into the Bronze raw layer.

However, Spark cannot directly analyze compressed claim files for our workflow.

Before loading data, we create an extraction folder.

Why we do this:

Raw ZIP file
        ↓
Extract contents
        ↓
Bronze ingestion
        ↓
Silver cleaning
        ↓
Gold analytics

This follows Databricks medallion architecture.

In [0]:
dbutils.fs.mkdirs(
    "dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/extracted/"
)

True

## Step 6 — Verify extraction folder

We created an extraction folder in the Bronze layer.

Now we confirm Databricks can see it.

Why:

Incorrect paths are one of the most common pipeline failures.

Verification before ingestion is standard practice.

In [0]:
display(
    dbutils.fs.ls(
        "dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/176541_DE1_0_2008_Beneficiary_Summary_File_Sample_1.zip,176541_DE1_0_2008_Beneficiary_Summary_File_Sample_1.zip,3119160,1779405806000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/extracted/,extracted/,0,1779406344709


## Step 7 — Extract CMS beneficiary file

The CMS dataset was uploaded as a ZIP archive.

Before loading into Spark, we need to extract the contents.

Why:

ZIP

 ↓

CSV/TXT file

 ↓

Bronze ingestion

 ↓

Silver cleaning

 ↓

Gold analytics

Claims pipelines usually start by extracting raw files.

In [0]:
import zipfile

zip_path = "/Volumes/healthcare_catalog/bronze/raw_claims_files/176541_DE1_0_2008_Beneficiary_Summary_File_Sample_1.zip"

extract_path = "/Volumes/healthcare_catalog/bronze/raw_claims_files/extracted/"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print("Extraction completed")

Extraction completed


## Step 8 — Inspect extracted CMS files

The ZIP file has been extracted.

Now we check which files were created.

Why:

Claims datasets may contain:
- CSV files
- TXT files
- documentation files

Understanding raw contents is always the first step before ingestion.

In [0]:
display(
    dbutils.fs.ls(
        "dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/extracted/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/extracted/DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv,DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv,14588413,1779406411000


## Step 9 — Load 2008 beneficiary CSV into Spark

Now we load the extracted CMS beneficiary CSV file.

This file represents Medicare beneficiaries.

In claims analytics, this is similar to the patient/member population table.

We are not cleaning yet.

Current goal:
Only load and preview the raw file.

In [0]:
beneficiary_2008_path = "dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/extracted/DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv"

beneficiary_2008_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(beneficiary_2008_path)
)

display(beneficiary_2008_df.limit(10))

DESYNPUF_ID,BENE_BIRTH_DT,BENE_DEATH_DT,BENE_SEX_IDENT_CD,BENE_RACE_CD,BENE_ESRD_IND,SP_STATE_CODE,BENE_COUNTY_CD,BENE_HI_CVRAGE_TOT_MONS,BENE_SMI_CVRAGE_TOT_MONS,BENE_HMO_CVRAGE_TOT_MONS,PLAN_CVRG_MOS_NUM,SP_ALZHDMTA,SP_CHF,SP_CHRNKIDN,SP_CNCR,SP_COPD,SP_DEPRESSN,SP_DIABETES,SP_ISCHMCHT,SP_OSTEOPRS,SP_RA_OA,SP_STRKETIA,MEDREIMB_IP,BENRES_IP,PPPYMT_IP,MEDREIMB_OP,BENRES_OP,PPPYMT_OP,MEDREIMB_CAR,BENRES_CAR,PPPYMT_CAR
00013D2EFD8E45D1,19230501,null,1,1,0,26,950,12,12,12,12,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,50.0,10.0,0.0,0.0,0.0,0.0
00016F745862898F,19430101,null,1,1,0,39,230,12,12,0,0,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0,700.0,240.0,0.0
0001FDD721E223DC,19360901,null,2,1,0,39,280,12,12,0,12,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
00021CA6FF03E670,19410601,null,1,5,0,6,290,0,0,0,0,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
00024B3D2352D2D0,19360801,null,1,1,0,52,590,12,12,0,0,2,2,2,2,2,2,2,2,1,2,2,0.0,0.0,0.0,30.0,40.0,0.0,220.0,80.0,0.0
0002DAE1C81CC70D,19431001,null,1,2,0,33,400,0,0,0,0,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0002F28CE057345B,19220701,null,1,1,0,39,270,12,12,0,0,2,1,1,2,2,1,1,1,2,1,2,0.0,0.0,0.0,1010.0,270.0,0.0,3330.0,940.0,0.0
000308435E3E5B76,19350901,null,1,1,0,24,680,10,10,0,0,2,2,2,2,2,2,2,2,1,2,2,0.0,0.0,0.0,150.0,160.0,0.0,870.0,340.0,80.0
000345A39D4157C9,19760901,null,2,1,0,23,810,0,0,0,0,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
00036A21B65B0206,19381001,null,2,2,0,1,570,12,12,12,12,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Step 10 — Understand key beneficiary columns

This file is the beneficiary/member table.

Important columns:

- DESYNPUF_ID: synthetic patient/member ID
- BENE_BIRTH_DT: beneficiary birth date
- BENE_DEATH_DT: death date, if available
- BENE_SEX_IDENT_CD: sex code
- BENE_RACE_CD: race code
- SP_STATE_CODE: state code
- BENE_COUNTY_CD: county code
- Chronic condition flags:
  - SP_ALZHDMTA
  - SP_CHF
  - SP_CHRNKIDN
  - SP_CNCR
  - SP_COPD
  - SP_DEPRESSN
  - SP_DIABETES
  - SP_ISCHMCHT
  - SP_OSTEOPRS
  - SP_RA_OA
  - SP_STRKETIA

Cost-related columns:

- MEDREIMB_IP: Medicare inpatient reimbursement
- BENRES_IP: beneficiary inpatient responsibility
- PPPYMT_IP: primary payer inpatient payment
- MEDREIMB_OP: Medicare outpatient reimbursement
- BENRES_OP: beneficiary outpatient responsibility
- PPPYMT_OP: primary payer outpatient payment
- MEDREIMB_CAR: Medicare carrier reimbursement
- BENRES_CAR: beneficiary carrier responsibility
- PPPYMT_CAR: primary payer carrier payment

This means this one file already contains basic cost burden information.

In [0]:
beneficiary_2008_df.printSchema()

root
 |-- DESYNPUF_ID: string (nullable = true)
 |-- BENE_BIRTH_DT: integer (nullable = true)
 |-- BENE_DEATH_DT: integer (nullable = true)
 |-- BENE_SEX_IDENT_CD: integer (nullable = true)
 |-- BENE_RACE_CD: integer (nullable = true)
 |-- BENE_ESRD_IND: string (nullable = true)
 |-- SP_STATE_CODE: integer (nullable = true)
 |-- BENE_COUNTY_CD: integer (nullable = true)
 |-- BENE_HI_CVRAGE_TOT_MONS: integer (nullable = true)
 |-- BENE_SMI_CVRAGE_TOT_MONS: integer (nullable = true)
 |-- BENE_HMO_CVRAGE_TOT_MONS: integer (nullable = true)
 |-- PLAN_CVRG_MOS_NUM: integer (nullable = true)
 |-- SP_ALZHDMTA: integer (nullable = true)
 |-- SP_CHF: integer (nullable = true)
 |-- SP_CHRNKIDN: integer (nullable = true)
 |-- SP_CNCR: integer (nullable = true)
 |-- SP_COPD: integer (nullable = true)
 |-- SP_DEPRESSN: integer (nullable = true)
 |-- SP_DIABETES: integer (nullable = true)
 |-- SP_ISCHMCHT: integer (nullable = true)
 |-- SP_OSTEOPRS: integer (nullable = true)
 |-- SP_RA_OA: integer (

## Step 11 — Count beneficiaries

Before performing claims analytics, we first determine population size.

Why:

Population count is one of the first KPIs in healthcare analytics.

Examples:

- Total covered members
- Total beneficiaries
- Total patients

This becomes denominator information for utilization metrics later.

In [0]:
beneficiary_2008_df.count()

116352

## Step 12 — Save beneficiary data to Bronze layer

Current status:

Raw CSV
    ↓
Spark DataFrame

Next:

Spark DataFrame
    ↓
Bronze table

Why:

Bronze tables preserve raw ingested data.

We avoid repeatedly loading CSV files.

In [0]:
beneficiary_2008_df.write \
    .mode("overwrite") \
    .saveAsTable(
        "healthcare_catalog.bronze.cms_beneficiary_2008_raw"
    )

## Step 13 — Verify Bronze table

Now we confirm that the Bronze table was saved correctly.

Why:

After writing any table, we always read it back and check the data.

This prevents silent pipeline mistakes.

In [0]:
%sql
SELECT *
FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw
LIMIT 10;

DESYNPUF_ID,BENE_BIRTH_DT,BENE_DEATH_DT,BENE_SEX_IDENT_CD,BENE_RACE_CD,BENE_ESRD_IND,SP_STATE_CODE,BENE_COUNTY_CD,BENE_HI_CVRAGE_TOT_MONS,BENE_SMI_CVRAGE_TOT_MONS,BENE_HMO_CVRAGE_TOT_MONS,PLAN_CVRG_MOS_NUM,SP_ALZHDMTA,SP_CHF,SP_CHRNKIDN,SP_CNCR,SP_COPD,SP_DEPRESSN,SP_DIABETES,SP_ISCHMCHT,SP_OSTEOPRS,SP_RA_OA,SP_STRKETIA,MEDREIMB_IP,BENRES_IP,PPPYMT_IP,MEDREIMB_OP,BENRES_OP,PPPYMT_OP,MEDREIMB_CAR,BENRES_CAR,PPPYMT_CAR
00013D2EFD8E45D1,19230501,null,1,1,0,26,950,12,12,12,12,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,50.0,10.0,0.0,0.0,0.0,0.0
00016F745862898F,19430101,null,1,1,0,39,230,12,12,0,0,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0,700.0,240.0,0.0
0001FDD721E223DC,19360901,null,2,1,0,39,280,12,12,0,12,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
00021CA6FF03E670,19410601,null,1,5,0,6,290,0,0,0,0,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
00024B3D2352D2D0,19360801,null,1,1,0,52,590,12,12,0,0,2,2,2,2,2,2,2,2,1,2,2,0.0,0.0,0.0,30.0,40.0,0.0,220.0,80.0,0.0
0002DAE1C81CC70D,19431001,null,1,2,0,33,400,0,0,0,0,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0002F28CE057345B,19220701,null,1,1,0,39,270,12,12,0,0,2,1,1,2,2,1,1,1,2,1,2,0.0,0.0,0.0,1010.0,270.0,0.0,3330.0,940.0,0.0
000308435E3E5B76,19350901,null,1,1,0,24,680,10,10,0,0,2,2,2,2,2,2,2,2,1,2,2,0.0,0.0,0.0,150.0,160.0,0.0,870.0,340.0,80.0
000345A39D4157C9,19760901,null,2,1,0,23,810,0,0,0,0,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
00036A21B65B0206,19381001,null,2,2,0,1,570,12,12,12,12,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Step 14 — Create sample beneficiary data for GitHub

The full CMS dataset is large.

For portfolio documentation and reproducibility,
we create a small sample dataset.

This sample helps:

- GitHub previews
- Portfolio demonstrations
- README examples
- Faster testing

In [0]:
beneficiary_sample_df = beneficiary_2008_df.limit(100)

display(beneficiary_sample_df)

DESYNPUF_ID,BENE_BIRTH_DT,BENE_DEATH_DT,BENE_SEX_IDENT_CD,BENE_RACE_CD,BENE_ESRD_IND,SP_STATE_CODE,BENE_COUNTY_CD,BENE_HI_CVRAGE_TOT_MONS,BENE_SMI_CVRAGE_TOT_MONS,BENE_HMO_CVRAGE_TOT_MONS,PLAN_CVRG_MOS_NUM,SP_ALZHDMTA,SP_CHF,SP_CHRNKIDN,SP_CNCR,SP_COPD,SP_DEPRESSN,SP_DIABETES,SP_ISCHMCHT,SP_OSTEOPRS,SP_RA_OA,SP_STRKETIA,MEDREIMB_IP,BENRES_IP,PPPYMT_IP,MEDREIMB_OP,BENRES_OP,PPPYMT_OP,MEDREIMB_CAR,BENRES_CAR,PPPYMT_CAR
00013D2EFD8E45D1,19230501,null,1,1,0,26,950,12,12,12,12,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,50.0,10.0,0.0,0.0,0.0,0.0
00016F745862898F,19430101,null,1,1,0,39,230,12,12,0,0,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0,700.0,240.0,0.0
0001FDD721E223DC,19360901,null,2,1,0,39,280,12,12,0,12,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
00021CA6FF03E670,19410601,null,1,5,0,6,290,0,0,0,0,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
00024B3D2352D2D0,19360801,null,1,1,0,52,590,12,12,0,0,2,2,2,2,2,2,2,2,1,2,2,0.0,0.0,0.0,30.0,40.0,0.0,220.0,80.0,0.0
0002DAE1C81CC70D,19431001,null,1,2,0,33,400,0,0,0,0,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0002F28CE057345B,19220701,null,1,1,0,39,270,12,12,0,0,2,1,1,2,2,1,1,1,2,1,2,0.0,0.0,0.0,1010.0,270.0,0.0,3330.0,940.0,0.0
000308435E3E5B76,19350901,null,1,1,0,24,680,10,10,0,0,2,2,2,2,2,2,2,2,1,2,2,0.0,0.0,0.0,150.0,160.0,0.0,870.0,340.0,80.0
000345A39D4157C9,19760901,null,2,1,0,23,810,0,0,0,0,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
00036A21B65B0206,19381001,null,2,2,0,1,570,12,12,12,12,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Step 15 — Save sample data for GitHub

We create a small CSV sample from CMS data.

Purpose:

- GitHub repository examples
- README screenshots
- Portfolio demonstrations
- Faster testing

Industry practice:

Keep sample data in GitHub,
keep full raw data outside GitHub.

In [0]:
sample_path = "/Volumes/healthcare_catalog/bronze/raw_claims_files/cms_beneficiary_sample.csv"

(
    beneficiary_sample_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(sample_path)
)

print("Sample file saved")

Sample file saved


## Step 16 — Locate exported sample CSV

Spark writes CSV output as a folder.

Inside the folder, there will be a part file.

We need to find that file before downloading it for GitHub.

In [0]:
display(
    dbutils.fs.ls(
        "dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/cms_beneficiary_sample.csv/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/cms_beneficiary_sample.csv/_SUCCESS,_SUCCESS,0,1779407243000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/cms_beneficiary_sample.csv/_committed_5122734322004558024,_committed_5122734322004558024,113,1779407242000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/cms_beneficiary_sample.csv/_started_5122734322004558024,_started_5122734322004558024,0,1779407242000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/cms_beneficiary_sample.csv/part-00000-tid-5122734322004558024-a52210f7-fa7d-4160-a30e-2e87756f5f6a-257-1-c000.csv,part-00000-tid-5122734322004558024-a52210f7-fa7d-4160-a30e-2e87756f5f6a-257-1-c000.csv,11890,1779407242000


## Step 17 — Rename sample CSV for GitHub

Spark creates a long part-file name.

For GitHub, we want a clean filename:

cms_beneficiary_2008_sample.csv

This makes the repository easier to understand.

In [0]:
source_file = "dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/cms_beneficiary_sample.csv/part-00000-tid-5122734322004558024-a52210f7-fa7d-4160-a30e-2e87756f5f6a-257-1-c000.csv"

target_file = "dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/cms_beneficiary_2008_sample.csv"

dbutils.fs.cp(source_file, target_file)

print("Sample CSV renamed for GitHub")

Sample CSV renamed for GitHub


## Step 18 — Verify GitHub sample dataset

Before downloading, verify the final sample file exists.

Expected file:

cms_beneficiary_2008_sample.csv

This will be uploaded to GitHub under:

data/sample/

In [0]:
display(
    dbutils.fs.ls(
        "dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/176541_DE1_0_2008_Beneficiary_Summary_File_Sample_1.zip,176541_DE1_0_2008_Beneficiary_Summary_File_Sample_1.zip,3119160,1779405806000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/cms_beneficiary_2008_sample.csv,cms_beneficiary_2008_sample.csv,11890,1779407349000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/cms_beneficiary_sample.csv/,cms_beneficiary_sample.csv/,0,1779407405386
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/extracted/,extracted/,0,1779407405386


## Step 19 — Download sample dataset for GitHub

We created a small CMS sample file.

Now we generate a downloadable link.

This sample dataset will be committed to GitHub.

In [0]:
displayHTML(
    """
    <a href="/files/Volumes/healthcare_catalog/bronze/raw_claims_files/cms_beneficiary_2008_sample.csv">
        Download CMS Sample CSV
    </a>
    """
)

Download CMS Sample CSV

## Step 21 — Calculate average inpatient reimbursement

Question:

How much Medicare reimburses on average for inpatient care?

Column:

MEDREIMB_IP

Meaning:

Medicare inpatient reimbursement amount.

Why important:

Claims analysts, payer analysts, HEOR teams, and healthcare data scientists often calculate average reimbursement metrics.

This becomes a cost KPI.

In [0]:
%sql
SELECT
    ROUND(
        AVG(MEDREIMB_IP),
        2
    ) AS avg_inpatient_reimbursement
FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw;

avg_inpatient_reimbursement
2214.18


## Interpretation — Average Inpatient Reimbursement

Result:

Average inpatient reimbursement = $2,214.18

Meaning:

On average, Medicare reimbursed approximately $2.2K per beneficiary for inpatient services in this CMS sample.

Possible inpatient services include:

- Hospital admission
- Surgery
- Emergency hospitalization
- Intensive treatment

Business interpretation:

Higher inpatient reimbursement often indicates:

- Increased hospitalization burden
- More severe illness
- Higher healthcare spending

Industry relevance:

This metric is commonly used in:

- Claims analytics
- Hospital finance
- Payer analytics
- HEOR
- Cost/utilization studies
- Population health analytics

Interview takeaway:

If asked:

"What is reimbursement analysis?"

You can explain:

"We calculate average reimbursement amounts to understand healthcare spending patterns and utilization burden."

## Step 22 — Calculate average outpatient reimbursement

Question:

How much Medicare reimburses on average for outpatient care?

Column:

MEDREIMB_OP

Meaning:

Medicare outpatient reimbursement.

Examples:

- Clinic visits
- Outpatient procedures
- Diagnostics
- Follow-up care

Why important:

Comparing inpatient vs outpatient reimbursement helps understand utilization and cost distribution.

In [0]:
%sql
SELECT
    ROUND(
        AVG(MEDREIMB_OP),
        2
    ) AS avg_outpatient_reimbursement
FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw;

avg_outpatient_reimbursement
622.23


## Interpretation — Average Outpatient Reimbursement

Result:

Average outpatient reimbursement = $622.23

Meaning:

On average, Medicare reimbursed approximately $622 for outpatient care.

Examples of outpatient services:

- Physician visits
- Diagnostics
- Imaging
- Follow-up appointments
- Minor procedures

Comparison with inpatient reimbursement:

Average inpatient reimbursement:
$2,214.18

Average outpatient reimbursement:
$622.23

Observation:

Inpatient reimbursement is much higher than outpatient reimbursement.

This is expected because inpatient care usually involves:

- Hospital stays
- Severe illness
- Intensive treatment
- Surgery

Business interpretation:

If outpatient reimbursement increases over time while inpatient decreases:

Possible interpretation:

Healthcare system shifting toward preventive or lower-cost care.

Industry relevance:

Used in:

- Utilization analytics
- Cost burden analysis
- Population health
- HEOR
- Payer analytics
- Hospital finance

Interview takeaway:

Comparing inpatient and outpatient reimbursement helps identify healthcare spending patterns and utilization trends.

## Step 23 — Calculate total inpatient Medicare spending

Question:

What is the total Medicare inpatient reimbursement across all beneficiaries?

Column:

MEDREIMB_IP

Why important:

Average cost shows typical spending.

Total cost shows overall financial burden.

Total spending is a common KPI in claims dashboards.

In [0]:
%sql
SELECT
    ROUND(
        SUM(MEDREIMB_IP),
        2
    ) AS total_inpatient_reimbursement
FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw;

total_inpatient_reimbursement
2.5762436E8


## Interpretation — Total Inpatient Reimbursement

Result:

Total inpatient reimbursement = $257,624,360

Meaning:

Across all beneficiaries in this CMS sample,
Medicare reimbursed approximately $257.6M
for inpatient services.

This measures:

Overall hospitalization-related spending burden.

Difference between average and total:

Average reimbursement:
Typical spending per beneficiary

Total reimbursement:
Overall healthcare financial burden

Business interpretation:

Higher total inpatient spending may indicate:

- Higher hospitalization rates
- Older populations
- Greater chronic disease burden
- Severe illness burden

Industry relevance:

Used in:

- Claims analytics
- Cost analytics
- Hospital finance
- HEOR
- Population health
- Payer analytics
- Medicare spending analysis

Interview takeaway:

Total reimbursement measures healthcare cost burden at population level.

## Step 24 — Total outpatient Medicare spending

Question:

What is total outpatient reimbursement?

Why:

Comparing inpatient and outpatient spending
helps understand utilization patterns and
cost distribution.

In [0]:
%sql
SELECT
    ROUND(
        SUM(MEDREIMB_OP),
        2
    ) AS total_outpatient_reimbursement
FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw;

total_outpatient_reimbursement
7.23973E7


## Interpretation — Total Outpatient Reimbursement

Result:

Total outpatient reimbursement = $72,397,300

Meaning:

Medicare reimbursed approximately $72.4M
for outpatient services across this population.

Comparison:

Total inpatient reimbursement:
$257.6M

Total outpatient reimbursement:
$72.4M

Observation:

Inpatient spending is much higher.

Possible reasons:

- Hospital admissions
- Severe illness
- Surgery
- Long stays
- Complex treatment

Business interpretation:

This suggests inpatient care contributes more heavily
to healthcare cost burden than outpatient care.

Healthcare organizations often target:

Reducing avoidable hospitalizations

because inpatient services are expensive.

Industry relevance:

Used in:

- Cost analytics
- Population health
- Claims analytics
- HEOR
- Hospital finance
- Value-based care studies

Interview takeaway:

Comparing inpatient and outpatient spending helps identify major healthcare cost drivers.

## Step 25 — Estimate inpatient utilization

Question:

How many beneficiaries had inpatient reimbursement > 0?

Interpretation:

If inpatient reimbursement exists,
the beneficiary likely used hospital inpatient services.

This becomes a utilization metric.

In [0]:
%sql
SELECT
    COUNT(*) AS inpatient_users
FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw
WHERE MEDREIMB_IP > 0;

inpatient_users
15422


## Interpretation — Inpatient Utilization

Result:

Beneficiaries with inpatient reimbursement > 0:

15,422

Meaning:

Approximately 15,422 Medicare beneficiaries used inpatient hospital services.

Examples:

- Hospital admissions
- Surgery
- Severe illness treatment
- Extended hospitalization

Comparison with total population:

Total beneficiaries:
116,352

Inpatient users:
15,422

Initial observation:

Only a subset of beneficiaries required inpatient care.

This is expected because hospitalization is relatively infrequent compared to outpatient care.

Business interpretation:

Higher inpatient utilization may indicate:

- Older populations
- Greater chronic disease burden
- Higher severity illness
- Increased healthcare spending

Industry relevance:

Used in:

- Utilization analytics
- Population health
- Claims analytics
- Hospital operations
- Cost forecasting
- HEOR

Interview takeaway:

Utilization metrics estimate how frequently healthcare services are used.

## Step 26 — Calculate inpatient utilization rate

Question:

What percentage of beneficiaries used inpatient services?

Formula:

Inpatient users
------------------------- × 100
Total beneficiaries

Why important:

Rates are more useful than counts when comparing populations.

In [0]:
%sql
SELECT
ROUND(
    100.0 *
    SUM(
        CASE
            WHEN MEDREIMB_IP > 0 THEN 1
            ELSE 0
        END
    )
    /
    COUNT(*)
,2) AS inpatient_utilization_rate_percent

FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw;

inpatient_utilization_rate_percent
13.25


## Interpretation — Inpatient Utilization Rate

Result:

Inpatient utilization rate = 13.25%

Meaning:

Approximately 13.25% of Medicare beneficiaries
used inpatient hospital services during this period.

Interpretation:

Out of every 100 beneficiaries:

~13 required hospitalization

~87 did not

Business interpretation:

Higher utilization rates may indicate:

- Older populations
- More chronic disease burden
- Severe illness burden
- Increased healthcare spending

Lower rates may indicate:

- Healthier populations
- Better preventive care
- Lower hospitalization burden

Industry relevance:

This KPI is used in:

- Population health analytics
- Hospital operations
- Claims analytics
- Utilization management
- Medicare analytics
- HEOR
- Value-based care

Interview takeaway:

Utilization rate measures how frequently healthcare services are used within a population.

Counts tell volume.

Rates tell burden.

## Step 27 — Identify high-cost beneficiaries

Question:

How many beneficiaries had inpatient reimbursement greater than $10,000?

Why important:

Healthcare systems often focus on:

High utilizers

High-cost patients

because a small group may drive large spending.

In [0]:
%sql
SELECT
COUNT(*) AS high_cost_beneficiaries
FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw
WHERE MEDREIMB_IP > 10000;

high_cost_beneficiaries
7933


## Interpretation — High-Cost Beneficiaries

Result:

Beneficiaries with inpatient reimbursement > $10,000

Count:

7,933

Meaning:

Approximately 7,933 beneficiaries generated very high inpatient costs.

These patients may represent:

- Frequent hospitalization
- Severe chronic illness
- Complex treatment
- Multiple comorbidities
- Long hospital stays

Important healthcare concept:

A small percentage of patients often contributes
to a large percentage of healthcare spending.

This is called:

High-cost high-need population

Business interpretation:

Healthcare organizations target these populations for:

- Care management
- Early intervention
- Predictive modeling
- Readmission reduction
- Cost reduction programs

Industry relevance:

Used heavily in:

- HEOR
- Claims analytics
- Population health
- Medicare analytics
- Healthcare AI
- Risk stratification

Interview takeaway:

Identifying high-cost patients is an important step before building cost prediction models.

## Step 28 — Calculate high-cost beneficiary rate

Question:

What percentage of beneficiaries had inpatient reimbursement > $10,000?

Why important:

Rates allow comparison across populations.

In [0]:
%sql
SELECT

ROUND(
100.0 *
SUM(
CASE
WHEN MEDREIMB_IP > 10000 THEN 1
ELSE 0
END
)
/ COUNT(*)
,2)

AS high_cost_rate_percent

FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw;

high_cost_rate_percent
6.82


## Interpretation — High-Cost Beneficiary Rate

Result:

High-cost beneficiary rate = 6.82%

Definition:

Beneficiaries with inpatient reimbursement > $10,000

Meaning:

Approximately 6.82% of beneficiaries belong to a high-cost population.

Interpretation:

Out of every 100 beneficiaries:

~7 are high-cost

~93 are not

Important healthcare insight:

A relatively small patient group often drives a disproportionate share of healthcare spending.

This concept appears frequently in:

- Medicare analytics
- Population health
- Healthcare AI
- Cost prediction
- Risk stratification
- HEOR
- Claims analytics

Business implication:

Healthcare organizations often focus interventions on these patients because reducing costs for a small high-cost group can produce large savings.

Interview takeaway:

High-cost population identification is commonly used before building:

- readmission prediction
- cost prediction
- hospitalization risk models
- care management programs

## Step 29 — Measure diabetes prevalence

Question:

How many beneficiaries have diabetes?

Why important:

Chronic diseases strongly influence:

- utilization
- hospitalization
- reimbursement
- healthcare spending

Diabetes is commonly analyzed in:

- Claims analytics
- Population health
- HEOR
- Cost studies

In [0]:
%sql
SELECT
SP_DIABETES,
COUNT(*) AS beneficiaries

FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw

GROUP BY SP_DIABETES
ORDER BY SP_DIABETES;

SP_DIABETES,beneficiaries
1,44060
2,72292


## Interpretation — Diabetes Burden

Result:

Beneficiaries with diabetes:
44,060

Beneficiaries without diabetes:
72,292

Meaning:

Approximately 44K beneficiaries have diabetes.

Observation:

Diabetes appears common in this Medicare population.

Why this matters:

Diabetes is associated with:

- higher utilization
- hospitalization
- medication burden
- chronic complications
- increased healthcare spending

Industry relevance:

Diabetes prevalence is frequently analyzed in:

- Population health
- Claims analytics
- HEOR
- Risk stratification
- Cost prediction
- Value-based care

Interview takeaway:

Chronic disease prevalence often becomes an explanatory variable for healthcare cost models.

## Step 30 — Calculate diabetes prevalence rate

Question:

What percentage of beneficiaries have diabetes?

Why important:

Disease prevalence is a common population health metric.

Formula:

People with diabetes
------------------------- × 100
Total population

In [0]:
%sql
SELECT

ROUND(
100.0 *
SUM(
CASE
WHEN SP_DIABETES = 1 THEN 1
ELSE 0
END
)
/ COUNT(*)
,2)

AS diabetes_prevalence_percent

FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw;

diabetes_prevalence_percent
37.87


## Interpretation — Diabetes Prevalence

Result:

Diabetes prevalence = 37.87%

Meaning:

Approximately 38 out of every 100 Medicare beneficiaries
have diabetes.

Observation:

This indicates a substantial chronic disease burden.

Why important:

Diabetes often increases:


- healthcare utilization
- medication use
- hospitalization risk
- complications
- healthcare spending

Potential downstream impact:

Higher prevalence may contribute to:

- increased reimbursement
- higher inpatient utilization
- greater cost burden

Industry relevance:

Used in:

- Population health
- Claims analytics
- HEOR
- Chronic disease studies
- Medicare analytics
- Risk prediction

Interview takeaway:

Disease prevalence helps quantify chronic illness burden in a population.

## Step 31 — Compare inpatient reimbursement by diabetes status

Question:

Do beneficiaries with diabetes have higher inpatient reimbursement?

Why important:

This explores relationships between:

Disease burden → healthcare cost

This is foundational for:

- HEOR
- Cost burden analysis
- Outcomes research
- Claims analytics

In [0]:
%sql
SELECT

SP_DIABETES,

ROUND(
AVG(MEDREIMB_IP),
2
)

AS avg_inpatient_reimbursement

FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw

GROUP BY SP_DIABETES

ORDER BY SP_DIABETES;

SP_DIABETES,avg_inpatient_reimbursement
1,4885.42
2,586.13


## Interpretation — Diabetes and Inpatient Cost Burden

Results:

Beneficiaries with diabetes:
Average inpatient reimbursement = $4,885.42

Beneficiaries without diabetes:
Average inpatient reimbursement = $586.13

Observation:

Beneficiaries with diabetes have much higher inpatient reimbursement.

Approximate comparison:

$4,885 ÷ $586 ≈ 8×

Meaning:

Patients with diabetes appear to generate substantially higher inpatient healthcare spending.

Possible reasons:

- Complications
- Frequent hospitalization
- Comorbidities
- Chronic disease management burden

HEOR interpretation:

This is an example of:

Disease burden → Increased healthcare cost

This type of analysis appears in:

- HEOR
- Cost burden studies
- Outcomes research
- Population health
- Claims analytics

Business implication:

Organizations may target diabetes prevention and management programs to reduce downstream hospitalization costs.

Interview takeaway:

Chronic diseases often act as predictors of increased healthcare utilization and spending.

## Step 32 — Calculate cost difference by diabetes status

Question:

How much additional inpatient reimbursement is associated with diabetes?

Why important:

HEOR frequently compares:

Disease group cost
vs
Non-disease group cost

This estimates economic burden.

In [0]:
%sql
WITH diabetes_cost AS (

SELECT
SP_DIABETES,
AVG(MEDREIMB_IP) AS avg_cost

FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw

GROUP BY SP_DIABETES

)

SELECT

ROUND(
MAX(CASE WHEN SP_DIABETES=1 THEN avg_cost END)
-
MAX(CASE WHEN SP_DIABETES=2 THEN avg_cost END)
,2)

AS additional_cost_associated_with_diabetes

FROM diabetes_cost;

additional_cost_associated_with_diabetes
4299.29


## Interpretation — Additional Cost Associated With Diabetes

Result:

Additional inpatient reimbursement associated with diabetes:

$4,299.29

Meaning:

On average, beneficiaries with diabetes generate approximately:

$4.3K more inpatient reimbursement

than beneficiaries without diabetes.

Interpretation:

This suggests diabetes is associated with substantially higher healthcare spending.

Potential contributors:

- Hospitalization
- Complications
- Chronic disease management
- Comorbidities

HEOR perspective:

This analysis estimates:

Economic burden of disease

Question answered:

"What additional healthcare cost is associated with diabetes?"

This is a foundational HEOR question.

Industry relevance:

Used in:

- Pharmaceutical HEOR
- Payer analytics
- Outcomes research
- Medicare studies
- Cost burden studies
- Health economics

Business implication:

Reducing diabetes complications could potentially reduce downstream inpatient spending.

Interview takeaway:

Disease-specific cost comparisons help estimate economic burden and support resource allocation decisions.

## Step 33 — Compare inpatient reimbursement by CHF status

Question:

Do beneficiaries with CHF have higher inpatient reimbursement?

Why important:

CHF is a major driver of:

- hospitalization
- readmission
- healthcare cost

Frequently studied in:

- HEOR
- Claims analytics
- Population health

In [0]:
%sql
SELECT

SP_CHF,

ROUND(
AVG(MEDREIMB_IP),
2
)

AS avg_inpatient_reimbursement

FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw

GROUP BY SP_CHF

ORDER BY SP_CHF;

SP_CHF,avg_inpatient_reimbursement
1,6408.96
2,542.51


## Interpretation — CHF and Inpatient Cost Burden

Results:

Beneficiaries with CHF:
Average inpatient reimbursement = $6,408.96

Beneficiaries without CHF:
Average inpatient reimbursement = $542.51

Observation:

Beneficiaries with CHF have dramatically higher inpatient reimbursement.

Approximate comparison:

$6,409 ÷ $543 ≈ 12×

Meaning:

Patients with CHF appear to generate substantially greater inpatient healthcare spending.

Potential reasons:

CHF often causes:

- repeated hospitalization
- acute episodes
- long stays
- severe complications
- readmissions

Comparison with diabetes:

Diabetes average inpatient reimbursement:
≈ $4,885

CHF average inpatient reimbursement:
≈ $6,409

Observation:

CHF appears associated with even greater inpatient spending burden.

HEOR interpretation:

CHF may represent a high-cost disease population.

Questions HEOR analysts ask:

How much economic burden does CHF create?

Can early intervention reduce cost?

Industry relevance:

Used in:

- Readmission studies
- Medicare analytics
- Outcomes research
- Population health
- Cost burden analysis
- Value-based care

Interview takeaway:

Certain chronic diseases drive disproportionate healthcare spending and become targets for intervention programs.

## Step 34 — Compare chronic disease cost burden

Question:

Between diabetes and CHF,
which appears associated with greater inpatient reimbursement burden?

Why important:

Disease prioritization helps:

- resource allocation
- intervention programs
- population health planning
- HEOR studies

In [0]:
%sql
SELECT
'Diabetes' AS condition,
ROUND(
AVG(
CASE
WHEN SP_DIABETES = 1
THEN MEDREIMB_IP
END
),2
) AS avg_inpatient_cost

FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw

UNION ALL

SELECT
'CHF',
ROUND(
AVG(
CASE
WHEN SP_CHF = 1
THEN MEDREIMB_IP
END
),2
)

FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw;

condition,avg_inpatient_cost
Diabetes,4885.42
CHF,6408.96


## Interpretation — Chronic Disease Cost Comparison

Results:

Diabetes:
Average inpatient reimbursement = $4,885.42

CHF:
Average inpatient reimbursement = $6,408.96

Observation:

CHF appears associated with greater inpatient cost burden than diabetes.

Approximate difference:

CHF − Diabetes

$6,409 − $4,885 ≈ $1,524

Meaning:

Beneficiaries with CHF may require more intensive and costly inpatient care.

Possible reasons:

CHF often leads to:

- hospitalization
- recurrent admissions
- severe complications
- long-term management burden

HEOR interpretation:

This suggests CHF may impose greater economic burden
than diabetes within this population.

Industry relevance:

Disease burden comparisons are used in:

- HEOR
- Outcomes research
- Cost burden studies
- Payer analytics
- Population health
- Resource allocation

Interview takeaway:

Comparing disease-specific costs helps prioritize interventions and identify high-cost populations.

## Step 35 — Compare cost burden across multiple chronic diseases

Question:

Which chronic diseases appear associated with the highest inpatient reimbursement?

Why important:

HEOR often identifies:

High-cost disease populations

to prioritize intervention strategies.

In [0]:
%sql
SELECT 'CHF' AS disease,
ROUND(AVG(CASE WHEN SP_CHF=1 THEN MEDREIMB_IP END),2)
AS avg_cost

UNION ALL

SELECT 'Diabetes',
ROUND(AVG(CASE WHEN SP_DIABETES=1 THEN MEDREIMB_IP END),2)

UNION ALL

SELECT 'COPD',
ROUND(AVG(CASE WHEN SP_COPD=1 THEN MEDREIMB_IP END),2)

UNION ALL

SELECT 'Cancer',
ROUND(AVG(CASE WHEN SP_CNCR=1 THEN MEDREIMB_IP END),2)

UNION ALL

SELECT 'Kidney Disease',
ROUND(AVG(CASE WHEN SP_CHRNKIDN=1 THEN MEDREIMB_IP END),2)

UNION ALL

SELECT 'Stroke/TIA',
ROUND(AVG(CASE WHEN SP_STRKETIA=1 THEN MEDREIMB_IP END),2)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7177429497791420>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "SELECT 'CHF' AS disease,\nROUND(AVG(CASE WHEN SP_CHF=1 THEN MEDREIMB_IP END),2)\nAS avg_cost\n\nUNION ALL\n\nSELECT 'Diabetes',\nROUND(AVG(CASE WHEN SP_DIABETES=1 THEN MEDREIMB_IP END),2)\n\nUNION ALL\n\nSELECT 'COPD',\nROUND(AVG(CASE WHEN SP_COPD=1 THEN MEDREIMB_IP END),2)\n\nUNION ALL\n\nSELECT 'Cancer',\nROUND(AVG(CASE WHEN SP_CNCR=1 THEN MEDREIMB_IP END),2)\n\nUNION ALL\n\nSELECT 'Kidney Disease',\nROUND(AVG(CASE WHEN SP_CHRNKIDN=1 THEN MEDREIMB_IP END),2)\n\nUNION ALL\n\nSELECT 'Stroke/TIA',\nROUND(AVG(CASE WHEN SP_STRKETIA=1 THEN MEDREIMB_IP END),2)\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   

## Step 35 — Compare chronic disease cost burden (corrected)

Question:

Which chronic diseases are associated with the highest inpatient reimbursement?

Purpose:

Identify high-cost chronic disease populations.

This is common in:

- HEOR
- Claims analytics
- Population health
- Cost burden studies

In [0]:
%sql
SELECT 'CHF' AS disease,
ROUND(
AVG(
CASE WHEN SP_CHF = 1
THEN MEDREIMB_IP
END
),2) AS avg_cost

FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw

UNION ALL

SELECT 'Diabetes',
ROUND(
AVG(
CASE WHEN SP_DIABETES=1
THEN MEDREIMB_IP
END
),2)

FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw

UNION ALL

SELECT 'COPD',
ROUND(
AVG(
CASE WHEN SP_COPD=1
THEN MEDREIMB_IP
END
),2)

FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw

UNION ALL

SELECT 'Cancer',
ROUND(
AVG(
CASE WHEN SP_CNCR=1
THEN MEDREIMB_IP
END
),2)

FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw

UNION ALL

SELECT 'Kidney Disease',
ROUND(
AVG(
CASE WHEN SP_CHRNKIDN=1
THEN MEDREIMB_IP
END
),2)

FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw

UNION ALL

SELECT 'Stroke/TIA',
ROUND(
AVG(
CASE WHEN SP_STRKETIA=1
THEN MEDREIMB_IP
END
),2)

FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw

disease,avg_cost
CHF,6408.96
Diabetes,4885.42
COPD,9793.18
Cancer,8288.89
Kidney Disease,9913.49
Stroke/TIA,12767.46


## Interpretation — Chronic Disease Cost Burden Ranking

Results:

Stroke/TIA      → $12,767
Kidney Disease  → $9,913
COPD            → $9,793
Cancer          → $8,289
CHF             → $6,409
Diabetes        → $4,885

Observation:

Stroke/TIA appears associated with the highest inpatient reimbursement burden.

Potential interpretation:

Stroke patients may experience:

- severe hospitalization
- long recovery
- recurrent admissions
- intensive treatment

Kidney disease and COPD also show substantial cost burden.

Important insight:

Not all chronic diseases contribute equally to healthcare spending.

HEOR perspective:

This analysis helps identify:

High-cost disease populations

which may become priorities for:

- intervention programs
- preventive care
- resource allocation
- care management

Business implication:

Organizations may focus on reducing costs among:

Stroke
Kidney disease
COPD

because they appear associated with higher spending.

Industry relevance:

Used in:

- HEOR
- Outcomes research
- Claims analytics
- Medicare studies
- Population health
- Payer analytics

Interview takeaway:

Ranking diseases by cost burden helps identify populations driving healthcare expenditure.

## Step 36 — Rank chronic diseases by cost burden

Question:

Which diseases have the greatest inpatient reimbursement burden?

Purpose:

Create an ordered disease burden ranking.

This resembles tables commonly seen in HEOR reports.

In [0]:
%sql
WITH disease_cost AS (

SELECT 'CHF' AS disease,
ROUND(AVG(CASE WHEN SP_CHF=1 THEN MEDREIMB_IP END),2)
AS avg_cost
FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw

UNION ALL

SELECT 'Diabetes',
ROUND(AVG(CASE WHEN SP_DIABETES=1 THEN MEDREIMB_IP END),2)
FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw

UNION ALL

SELECT 'COPD',
ROUND(AVG(CASE WHEN SP_COPD=1 THEN MEDREIMB_IP END),2)
FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw

UNION ALL

SELECT 'Cancer',
ROUND(AVG(CASE WHEN SP_CNCR=1 THEN MEDREIMB_IP END),2)
FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw

UNION ALL

SELECT 'Kidney Disease',
ROUND(AVG(CASE WHEN SP_CHRNKIDN=1 THEN MEDREIMB_IP END),2)
FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw

UNION ALL

SELECT 'Stroke/TIA',
ROUND(AVG(CASE WHEN SP_STRKETIA=1 THEN MEDREIMB_IP END),2)
FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw

)

SELECT *
FROM disease_cost
ORDER BY avg_cost DESC;

disease,avg_cost
Stroke/TIA,12767.46
Kidney Disease,9913.49
COPD,9793.18
Cancer,8288.89
CHF,6408.96
Diabetes,4885.42


## Interpretation — Disease Cost Burden Ranking

Objective:

Rank chronic diseases by average inpatient reimbursement burden.

Results:

1. Stroke/TIA       → $12,767
2. Kidney Disease   → $9,913
3. COPD             → $9,793
4. Cancer           → $8,289
5. CHF              → $6,409
6. Diabetes         → $4,885

Key finding:

Stroke/TIA appears associated with the highest inpatient cost burden.

Possible explanation:

Stroke often involves:

- emergency hospitalization
- rehabilitation
- recurrent complications
- long stays

Kidney disease and COPD also contribute substantial cost burden.

Population health implication:

Disease prevention for these conditions may significantly reduce spending.

HEOR implication:

These findings help estimate:

Disease-specific economic burden

Claims analytics implication:

These populations may become targets for:

- care management
- predictive models
- utilization reduction
- intervention programs

Interview takeaway:

Ranking diseases by cost burden supports prioritization of healthcare resources and cost reduction strategies.

## Step 37 — Save disease burden ranking to Gold layer

Purpose:

Store chronic disease cost burden results as a reusable analytics table.

Gold tables support:

- dashboards
- reporting
- ML features
- HEOR studies

In [0]:
%sql
CREATE OR REPLACE TABLE
healthcare_catalog.gold.disease_cost_burden_summary

AS

WITH disease_cost AS (

SELECT 'CHF' AS disease,
ROUND(AVG(CASE WHEN SP_CHF=1 THEN MEDREIMB_IP END),2)
AS avg_cost
FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw

UNION ALL

SELECT 'Diabetes',
ROUND(AVG(CASE WHEN SP_DIABETES=1 THEN MEDREIMB_IP END),2)
FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw

UNION ALL

SELECT 'COPD',
ROUND(AVG(CASE WHEN SP_COPD=1 THEN MEDREIMB_IP END),2)
FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw

UNION ALL

SELECT 'Cancer',
ROUND(AVG(CASE WHEN SP_CNCR=1 THEN MEDREIMB_IP END),2)
FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw

UNION ALL

SELECT 'Kidney Disease',
ROUND(AVG(CASE WHEN SP_CHRNKIDN=1 THEN MEDREIMB_IP END),2)
FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw

UNION ALL

SELECT 'Stroke/TIA',
ROUND(AVG(CASE WHEN SP_STRKETIA=1 THEN MEDREIMB_IP END),2)
FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw

)

SELECT *
FROM disease_cost
ORDER BY avg_cost DESC;

num_affected_rows,num_inserted_rows


## Step 38 — Verify Gold disease cost burden table

We created a Gold analytics table:

healthcare_catalog.gold.disease_cost_burden_summary

Now we read the table to confirm it saved correctly.

Why:

Gold tables should always be verified after creation.

In [0]:
%sql
SELECT *
FROM healthcare_catalog.gold.disease_cost_burden_summary;

disease,avg_cost
Stroke/TIA,12767.46
Kidney Disease,9913.49
COPD,9793.18
Cancer,8288.89
CHF,6408.96
Diabetes,4885.42


## Step 39 — Export Gold disease burden summary

Purpose:

Save Gold analytics output for:

- GitHub
- Portfolio documentation
- Dashboard examples
- README figures

Industry practice:

Share curated Gold outputs,
not full raw healthcare data.

In [0]:
gold_df = spark.table(
    "healthcare_catalog.gold.disease_cost_burden_summary"
)

(
    gold_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(
        "/Volumes/healthcare_catalog/gold/disease_cost_burden_summary_export"
    )
)

print("Gold table exported")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6890174881413657>, line 11
      1 gold_df = spark.table(
      2     "healthcare_catalog.gold.disease_cost_burden_summary"
      3 )
      5 (
      6     gold_df
      7     .coalesce(1)
      8     .write
      9     .mode("overwrite")
     10     .option("header", True)
---> 11     .csv(
     12         "/Volumes/healthcare_catalog/gold/disease_cost_burden_summary_export"
     13     )
     14 )
     16 print("Gold table exported")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:831, in DataFrameWriter.csv(self, path, mode, compression, sep, quote, escape, header, nullValue, escapeQuotes, quoteAll, dateFormat, timestampFormat, ignoreLeadingWhiteSpace, ignoreTrailingWhiteSpace, charToEscapeQuoteEscaping, encoding, emptyValue, lineSep)
    812 self.mode(mode)
    813 self._set_opts

## Step 39 — Export Gold disease burden summary for GitHub

Purpose:

Save curated analytics output for portfolio use.

This demonstrates:

Claims Analytics
+
HEOR
+
Gold Layer Engineering

In [0]:
gold_df = spark.table(
    "healthcare_catalog.gold.disease_cost_burden_summary"
)

(
    gold_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(
        "/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_cost_burden_summary_export"
    )
)

print("Gold table exported")

Gold table exported


## Step 40 — Locate exported Gold CSV file

Spark exports CSV as a folder.

Inside the folder, we need the part file.

We will rename it to:

disease_cost_burden_summary.csv

In [0]:
display(
    dbutils.fs.ls(
        "dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_cost_burden_summary_export/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_cost_burden_summary_export/_SUCCESS,_SUCCESS,0,1779410437000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_cost_burden_summary_export/_committed_7381306528141911606,_committed_7381306528141911606,113,1779410436000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_cost_burden_summary_export/_started_7381306528141911606,_started_7381306528141911606,0,1779410436000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_cost_burden_summary_export/part-00000-tid-7381306528141911606-5a163f7e-3036-4890-a8ec-f6466b6bc44d-347-1-c000.csv,part-00000-tid-7381306528141911606-5a163f7e-3036-4890-a8ec-f6466b6bc44d-347-1-c000.csv,117,1779410436000


## Step 41 — Rename Gold output for GitHub

Spark creates long part-file names.

For GitHub, we rename the file to:

disease_cost_burden_summary.csv

This file will go under:

analytics_outputs/

In [0]:
source_file = "dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_cost_burden_summary_export/part-00000-tid-7381306528141911606-5a163f7e-3036-4890-a8ec-f6466b6bc44d-347-1-c000.csv"

target_file = "dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_cost_burden_summary.csv"

dbutils.fs.cp(source_file, target_file)

print("Gold CSV renamed for GitHub")

Gold CSV renamed for GitHub


## Step 42 — Verify exported Gold analytics file

Before downloading for GitHub,
confirm the renamed Gold CSV exists.

Expected:

disease_cost_burden_summary.csv

In [0]:
display(
    dbutils.fs.ls(
        "dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/176541_DE1_0_2008_Beneficiary_Summary_File_Sample_1.zip,176541_DE1_0_2008_Beneficiary_Summary_File_Sample_1.zip,3119160,1779405806000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/cms_beneficiary_2008_sample.csv,cms_beneficiary_2008_sample.csv,11890,1779407349000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/cms_beneficiary_sample.csv/,cms_beneficiary_sample.csv/,0,1779410564176
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_cost_burden_summary.csv,disease_cost_burden_summary.csv,117,1779410525000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_cost_burden_summary_export/,disease_cost_burden_summary_export/,0,1779410564176
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/extracted/,extracted/,0,1779410564176


## Step 43 — Create Silver beneficiary table

Current issue:

We created a Bronze table and then directly created Gold analytics.

For proper medallion architecture, we should create a Silver table first.

Bronze table:

Raw ingested data.

Silver table:

Cleaned, standardized, analysis-ready data.

What we clean:

- Rename technical CMS columns into readable names
- Convert sex code into labels
- Convert race code into labels
- Convert chronic condition flags into 0/1 indicators
- Create total reimbursement columns
- Create high-cost flag

In [0]:
%sql
CREATE OR REPLACE TABLE healthcare_catalog.silver.cms_beneficiary_2008_clean AS

SELECT
    DESYNPUF_ID AS beneficiary_id,

    BENE_BIRTH_DT AS birth_date_raw,
    BENE_DEATH_DT AS death_date_raw,

    CASE
        WHEN BENE_SEX_IDENT_CD = 1 THEN 'Male'
        WHEN BENE_SEX_IDENT_CD = 2 THEN 'Female'
        ELSE 'Unknown'
    END AS sex,

    CASE
        WHEN BENE_RACE_CD = 1 THEN 'White'
        WHEN BENE_RACE_CD = 2 THEN 'Black'
        WHEN BENE_RACE_CD = 3 THEN 'Other'
        WHEN BENE_RACE_CD = 5 THEN 'Hispanic'
        ELSE 'Unknown'
    END AS race,

    SP_STATE_CODE AS state_code,
    BENE_COUNTY_CD AS county_code,

    CASE WHEN SP_DIABETES = 1 THEN 1 ELSE 0 END AS diabetes_flag,
    CASE WHEN SP_CHF = 1 THEN 1 ELSE 0 END AS chf_flag,
    CASE WHEN SP_COPD = 1 THEN 1 ELSE 0 END AS copd_flag,
    CASE WHEN SP_CNCR = 1 THEN 1 ELSE 0 END AS cancer_flag,
    CASE WHEN SP_CHRNKIDN = 1 THEN 1 ELSE 0 END AS kidney_disease_flag,
    CASE WHEN SP_STRKETIA = 1 THEN 1 ELSE 0 END AS stroke_tia_flag,

    MEDREIMB_IP AS inpatient_reimbursement,
    MEDREIMB_OP AS outpatient_reimbursement,
    MEDREIMB_CAR AS carrier_reimbursement,

    BENRES_IP AS inpatient_patient_responsibility,
    BENRES_OP AS outpatient_patient_responsibility,
    BENRES_CAR AS carrier_patient_responsibility,

    PPPYMT_IP AS inpatient_primary_payer_payment,
    PPPYMT_OP AS outpatient_primary_payer_payment,
    PPPYMT_CAR AS carrier_primary_payer_payment,

    COALESCE(MEDREIMB_IP,0)
    + COALESCE(MEDREIMB_OP,0)
    + COALESCE(MEDREIMB_CAR,0)
    AS total_medicare_reimbursement,

    CASE
        WHEN MEDREIMB_IP > 10000 THEN 1
        ELSE 0
    END AS high_inpatient_cost_flag

FROM healthcare_catalog.bronze.cms_beneficiary_2008_raw;

num_affected_rows,num_inserted_rows


## Step 44 — Verify Silver beneficiary table

We created a cleaned Silver table:

healthcare_catalog.silver.cms_beneficiary_2008_clean

Now we preview it to confirm that readable columns and engineered fields were created correctly.

In [0]:
%sql
SELECT *
FROM healthcare_catalog.silver.cms_beneficiary_2008_clean
LIMIT 10;

beneficiary_id,birth_date_raw,death_date_raw,sex,race,state_code,county_code,diabetes_flag,chf_flag,copd_flag,cancer_flag,kidney_disease_flag,stroke_tia_flag,inpatient_reimbursement,outpatient_reimbursement,carrier_reimbursement,inpatient_patient_responsibility,outpatient_patient_responsibility,carrier_patient_responsibility,inpatient_primary_payer_payment,outpatient_primary_payer_payment,carrier_primary_payer_payment,total_medicare_reimbursement,high_inpatient_cost_flag
00013D2EFD8E45D1,19230501,null,Male,White,26,950,0,0,0,0,0,0,0.0,50.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,50.0,0
00016F745862898F,19430101,null,Male,White,39,230,0,0,0,0,0,0,0.0,0.0,700.0,0.0,0.0,240.0,0.0,0.0,0.0,700.0,0
0001FDD721E223DC,19360901,null,Female,White,39,280,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
00021CA6FF03E670,19410601,null,Male,Hispanic,6,290,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
00024B3D2352D2D0,19360801,null,Male,White,52,590,0,0,0,0,0,0,0.0,30.0,220.0,0.0,40.0,80.0,0.0,0.0,0.0,250.0,0
0002DAE1C81CC70D,19431001,null,Male,Black,33,400,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
0002F28CE057345B,19220701,null,Male,White,39,270,1,1,0,0,1,0,0.0,1010.0,3330.0,0.0,270.0,940.0,0.0,0.0,0.0,4340.0,0
000308435E3E5B76,19350901,null,Male,White,24,680,0,0,0,0,0,0,0.0,150.0,870.0,0.0,160.0,340.0,0.0,0.0,80.0,1020.0,0
000345A39D4157C9,19760901,null,Female,White,23,810,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
00036A21B65B0206,19381001,null,Female,Black,1,570,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


## Interpretation — Silver Beneficiary Table

The Silver table successfully converts raw CMS beneficiary data into a cleaner analytics-ready format.

Important improvements:

- CMS technical columns were renamed into readable names
- Sex codes were converted into labels
- Race codes were converted into labels
- Chronic condition indicators were converted into 0/1 flags
- Reimbursement fields were standardized
- Total Medicare reimbursement was calculated
- High inpatient cost flag was created

Why this matters:

Bronze stores raw ingested data.

Silver makes data usable for analysis, reporting, dashboards, and machine learning.

This table will be used for:

- claims analytics
- cost/utilization analysis
- chronic disease burden
- HEOR analysis
- high-cost patient prediction

## Step 46 — Export Silver sample dataset

Purpose:

Save cleaned Silver data for:

- GitHub
- Portfolio documentation
- Medallion architecture demonstration

This proves:

Bronze → Silver transformation exists.

In [0]:
silver_df = spark.table(
    "healthcare_catalog.silver.cms_beneficiary_2008_clean"
)

(
    silver_df
    .limit(100)
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(
        "/Volumes/healthcare_catalog/bronze/raw_claims_files/silver_beneficiary_export"
    )
)

print("Silver sample exported")

Silver sample exported


## Step 47 — Locate Silver sample export

Spark writes CSV exports as folders.

We locate the part-file before renaming it for GitHub.

In [0]:
display(
    dbutils.fs.ls(
        "dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/silver_beneficiary_export/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/silver_beneficiary_export/_SUCCESS,_SUCCESS,0,1779411069000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/silver_beneficiary_export/_committed_8113721420861925083,_committed_8113721420861925083,113,1779411069000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/silver_beneficiary_export/_started_8113721420861925083,_started_8113721420861925083,0,1779411068000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/silver_beneficiary_export/part-00000-tid-8113721420861925083-5725419f-2454-460a-95ae-8a2f45f61459-363-1-c000.csv,part-00000-tid-8113721420861925083-5725419f-2454-460a-95ae-8a2f45f61459-363-1-c000.csv,11333,1779411069000


## Step 48 — Rename Silver sample file for GitHub

Spark creates long part-file names.

For GitHub, we rename it to:

cms_beneficiary_2008_clean_sample.csv

This file represents the Silver cleaned version of the CMS beneficiary data.

In [0]:
source_file = "dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/silver_beneficiary_export/part-00000-tid-8113721420861925083-5725419f-2454-460a-95ae-8a2f45f61459-363-1-c000.csv"

target_file = "dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/cms_beneficiary_2008_clean_sample.csv"

dbutils.fs.cp(source_file, target_file)

print("Silver sample renamed for GitHub")

Silver sample renamed for GitHub


## Step 49 — Verify renamed Silver sample file

Confirm the final cleaned Silver sample file exists.

Expected:

cms_beneficiary_2008_clean_sample.csv

In [0]:
display(
    dbutils.fs.ls(
        "dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/176541_DE1_0_2008_Beneficiary_Summary_File_Sample_1.zip,176541_DE1_0_2008_Beneficiary_Summary_File_Sample_1.zip,3119160,1779405806000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/cms_beneficiary_2008_clean_sample.csv,cms_beneficiary_2008_clean_sample.csv,11333,1779411237000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/cms_beneficiary_2008_sample.csv,cms_beneficiary_2008_sample.csv,11890,1779407349000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/cms_beneficiary_sample.csv/,cms_beneficiary_sample.csv/,0,1779411342141
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_cost_burden_summary.csv,disease_cost_burden_summary.csv,117,1779410525000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_cost_burden_summary_export/,disease_cost_burden_summary_export/,0,1779411342141
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/extracted/,extracted/,0,1779411342141
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/silver_beneficiary_export/,silver_beneficiary_export/,0,1779411342141


## Step 50 — Recreate Gold disease burden table using Silver layer

Purpose:

Rebuild Gold analytics from cleaned Silver data.

This creates proper medallion lineage:

Bronze
 ↓
Silver
 ↓
Gold

Gold tables should generally depend on Silver, not raw Bronze.

In [0]:
%sql
CREATE OR REPLACE TABLE
healthcare_catalog.gold.disease_cost_burden_summary

AS

WITH disease_cost AS (

SELECT
'Stroke/TIA' AS disease,
ROUND(
AVG(
CASE WHEN stroke_tia_flag=1
THEN inpatient_reimbursement
END
),2
) AS avg_cost

FROM healthcare_catalog.silver.cms_beneficiary_2008_clean

UNION ALL

SELECT
'Kidney Disease',
ROUND(
AVG(
CASE WHEN kidney_disease_flag=1
THEN inpatient_reimbursement
END
),2
)

FROM healthcare_catalog.silver.cms_beneficiary_2008_clean

UNION ALL

SELECT
'COPD',
ROUND(
AVG(
CASE WHEN copd_flag=1
THEN inpatient_reimbursement
END
),2
)

FROM healthcare_catalog.silver.cms_beneficiary_2008_clean

UNION ALL

SELECT
'Cancer',
ROUND(
AVG(
CASE WHEN cancer_flag=1
THEN inpatient_reimbursement
END
),2
)

FROM healthcare_catalog.silver.cms_beneficiary_2008_clean

UNION ALL

SELECT
'CHF',
ROUND(
AVG(
CASE WHEN chf_flag=1
THEN inpatient_reimbursement
END
),2
)

FROM healthcare_catalog.silver.cms_beneficiary_2008_clean

UNION ALL

SELECT
'Diabetes',
ROUND(
AVG(
CASE WHEN diabetes_flag=1
THEN inpatient_reimbursement
END
),2
)

FROM healthcare_catalog.silver.cms_beneficiary_2008_clean

)

SELECT *
FROM disease_cost
ORDER BY avg_cost DESC;

num_affected_rows,num_inserted_rows


## Step 51 — Verify Gold table rebuilt from Silver

We recreated the Gold disease burden table using the Silver cleaned table.

Now we verify the final output.

Expected:

The ranking should still show:

- Stroke/TIA
- Kidney Disease
- COPD
- Cancer
- CHF
- Diabetes

In [0]:
%sql
SELECT *
FROM healthcare_catalog.gold.disease_cost_burden_summary;

disease,avg_cost
Stroke/TIA,12767.46
Kidney Disease,9913.49
COPD,9793.18
Cancer,8288.89
CHF,6408.96
Diabetes,4885.42


## Interpretation — Corrected Gold Lineage

The disease cost burden summary table was rebuilt from the Silver layer.

Correct architecture:

Raw CMS file
    ↓
Bronze raw table
    ↓
Silver cleaned table
    ↓
Gold analytics table

This is better than building Gold directly from Bronze.

Why:

- Bronze preserves raw data
- Silver standardizes and cleans data
- Gold supports analytics, dashboards, and reporting

Final Gold output:

Stroke/TIA has the highest average inpatient reimbursement,
followed by kidney disease, COPD, cancer, CHF, and diabetes.

This Gold table is now ready for:

- dashboard development
- HEOR reporting
- portfolio documentation
- Power BI export

## Step 52 — Create disease prevalence summary table

Question:

How common is each chronic disease?

Purpose:

Measure population burden.

This table will support:

- dashboards
- population health
- HEOR
- claims analytics

We already measured diabetes prevalence.

Now we calculate prevalence for multiple diseases.

In [0]:
%sql
CREATE OR REPLACE TABLE
healthcare_catalog.gold.disease_prevalence_summary

AS

WITH prevalence AS (

SELECT
'Stroke/TIA' AS disease,
ROUND(
100.0 *
AVG(stroke_tia_flag)
,2
)
AS prevalence_percent

FROM healthcare_catalog.silver.cms_beneficiary_2008_clean

UNION ALL

SELECT
'Kidney Disease',
ROUND(
100.0 *
AVG(kidney_disease_flag)
,2
)

FROM healthcare_catalog.silver.cms_beneficiary_2008_clean

UNION ALL

SELECT
'COPD',
ROUND(
100.0 *
AVG(copd_flag)
,2
)

FROM healthcare_catalog.silver.cms_beneficiary_2008_clean

UNION ALL

SELECT
'Cancer',
ROUND(
100.0 *
AVG(cancer_flag)
,2
)

FROM healthcare_catalog.silver.cms_beneficiary_2008_clean

UNION ALL

SELECT
'CHF',
ROUND(
100.0 *
AVG(chf_flag)
,2
)

FROM healthcare_catalog.silver.cms_beneficiary_2008_clean

UNION ALL

SELECT
'Diabetes',
ROUND(
100.0 *
AVG(diabetes_flag)
,2
)

FROM healthcare_catalog.silver.cms_beneficiary_2008_clean

)

SELECT *
FROM prevalence
ORDER BY prevalence_percent DESC;

num_affected_rows,num_inserted_rows


## Step 53 — Verify disease prevalence summary

We created a Gold table:

healthcare_catalog.gold.disease_prevalence_summary

Now we read it to confirm disease prevalence results.

Prevalence means:

What percentage of beneficiaries have each disease?

In [0]:
%sql
SELECT *
FROM healthcare_catalog.gold.disease_prevalence_summary;

disease,prevalence_percent
Diabetes,37.87
CHF,28.5
Kidney Disease,16.06
COPD,13.53
Cancer,6.37
Stroke/TIA,4.49


## Interpretation — Disease Prevalence Ranking

Objective:

Measure chronic disease burden within the Medicare population.

Results:

Diabetes        → 37.87%
CHF             → 28.50%
Kidney Disease  → 16.06%
COPD            → 13.53%
Cancer          → 6.37%
Stroke/TIA      → 4.49%

Observation:

Diabetes is the most common chronic disease.

However:

Highest prevalence ≠ Highest cost burden

Example:

Diabetes:
Most common disease

Stroke/TIA:
Highest average inpatient reimbursement

Important healthcare insight:

Common diseases and expensive diseases are not always the same.

HEOR implication:

Two important questions:

Which diseases are common?

Which diseases are expensive?

Both matter for healthcare planning.

Interview takeaway:

Prevalence measures population burden.
Cost burden measures economic burden.

## Step 54 — Create dashboard-ready disease summary table

Goal:

Combine:

Disease prevalence
+
Disease cost burden

Purpose:

Create a single table for:

- dashboards
- Power BI
- HEOR reporting
- claims analytics

In [0]:
%sql
CREATE OR REPLACE TABLE
healthcare_catalog.gold.disease_dashboard_summary

AS

SELECT

p.disease,

p.prevalence_percent,

c.avg_cost AS avg_inpatient_cost

FROM healthcare_catalog.gold.disease_prevalence_summary p

LEFT JOIN
healthcare_catalog.gold.disease_cost_burden_summary c

ON p.disease = c.disease

ORDER BY avg_inpatient_cost DESC;

num_affected_rows,num_inserted_rows


## Step 55 — Verify dashboard-ready disease summary

We created a combined Gold table:

healthcare_catalog.gold.disease_dashboard_summary

This table combines:

- disease prevalence
- average inpatient cost

This is dashboard-ready and HEOR-report-ready.

In [0]:
%sql
SELECT *
FROM healthcare_catalog.gold.disease_dashboard_summary;

disease,prevalence_percent,avg_inpatient_cost
Stroke/TIA,4.49,12767.46
Kidney Disease,16.06,9913.49
COPD,13.53,9793.18
Cancer,6.37,8288.89
CHF,28.5,6408.96
Diabetes,37.87,4885.42


## Interpretation — Dashboard Disease Summary

This table combines:

1. Disease prevalence
2. Average inpatient cost burden

This allows comparison between:

Common diseases
vs
Expensive diseases

Key findings:

Diabetes:

- Highest prevalence (37.87%)
- Lower cost burden than several diseases

Stroke/TIA:

- Low prevalence (4.49%)
- Highest inpatient reimbursement burden

Important insight:

High prevalence does not always mean high cost.

Example:

Diabetes:
Common disease

Stroke:
Expensive disease

HEOR implication:

Healthcare systems may prioritize:

High prevalence diseases:
Population management

High cost diseases:
Cost reduction interventions

Dashboard implication:

This table is suitable for:

- Power BI
- Tableau
- Executive dashboards
- HEOR reporting
- Claims analytics reporting

Interview takeaway:

Combining prevalence and cost creates richer disease burden analysis.

## Step 56 — Export dashboard summary for GitHub

Purpose:

Save final dashboard-ready analytics output.

This demonstrates:

Claims Analytics
+
Population Health
+
HEOR
+
Gold Layer

In [0]:
dashboard_df = spark.table(
    "healthcare_catalog.gold.disease_dashboard_summary"
)

(
    dashboard_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(
        "/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_dashboard_summary_export"
    )
)

print("Dashboard summary exported")

Dashboard summary exported



## Step 57 — Locate dashboard summary export

Spark exports CSV as a folder.

Inside the folder, we need the part-file before renaming it for GitHub.

In [0]:
display(
    dbutils.fs.ls(
        "dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_dashboard_summary_export/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_dashboard_summary_export/_SUCCESS,_SUCCESS,0,1779453212000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_dashboard_summary_export/_committed_3078453298197033262,_committed_3078453298197033262,113,1779453212000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_dashboard_summary_export/_started_3078453298197033262,_started_3078453298197033262,0,1779453211000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_dashboard_summary_export/part-00000-tid-3078453298197033262-0f94c23a-9fbd-43e1-9e9f-f7c95b1816aa-240-1-c000.csv,part-00000-tid-3078453298197033262-0f94c23a-9fbd-43e1-9e9f-f7c95b1816aa-240-1-c000.csv,179,1779453211000


## Step 58 — Rename dashboard summary file

Spark creates long part-file names.

For GitHub, rename to:

disease_dashboard_summary.csv

This is the final dashboard-ready Gold output.

In [0]:
source_file = "dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_dashboard_summary_export/part-00000-tid-3078453298197033262-0f94c23a-9fbd-43e1-9e9f-f7c95b1816aa-240-1-c000.csv"

target_file = "dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_dashboard_summary.csv"

dbutils.fs.cp(source_file, target_file)

print("Dashboard summary renamed for GitHub")

Dashboard summary renamed for GitHub


## Step 59 — Verify final dashboard output

Confirm the renamed dashboard summary file exists.

Expected:

disease_dashboard_summary.csv

In [0]:
display(
    dbutils.fs.ls(
        "dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/176541_DE1_0_2008_Beneficiary_Summary_File_Sample_1.zip,176541_DE1_0_2008_Beneficiary_Summary_File_Sample_1.zip,3119160,1779405806000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/cms_beneficiary_2008_clean_sample.csv,cms_beneficiary_2008_clean_sample.csv,11333,1779411237000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/cms_beneficiary_2008_sample.csv,cms_beneficiary_2008_sample.csv,11890,1779407349000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/cms_beneficiary_sample.csv/,cms_beneficiary_sample.csv/,0,1779453656642
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_cost_burden_summary.csv,disease_cost_burden_summary.csv,117,1779410525000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_cost_burden_summary_export/,disease_cost_burden_summary_export/,0,1779453656642
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_dashboard_summary.csv,disease_dashboard_summary.csv,179,1779453520000
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/disease_dashboard_summary_export/,disease_dashboard_summary_export/,0,1779453656642
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/extracted/,extracted/,0,1779453656642
dbfs:/Volumes/healthcare_catalog/bronze/raw_claims_files/silver_beneficiary_export/,silver_beneficiary_export/,0,1779453656642
